# 02 — Fine-Tuning Walkthrough

**Key question**: *How much does LoRA reduce compute cost without sacrificing performance?*

This notebook walks through the full training pipeline: ETL, config loading, backbone instantiation, LoRA wrapping, class weight computation, and the training run with loss curves.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import torch
from omegaconf import OmegaConf
from transformers import AutoTokenizer, TrainingArguments

from dissatisfaction_classifier.data.dataset import DissatisfactionDataset
from dissatisfaction_classifier.data.validation import validate_reviews_df
from dissatisfaction_classifier.models.backbone import load_backbone
from dissatisfaction_classifier.models.lora_wrapper import apply_lora
from dissatisfaction_classifier.training.trainer import WeightedLossTrainer, compute_class_weights
from dissatisfaction_classifier.training.callbacks import WandbMetricsCallback
from dissatisfaction_classifier.evaluation.metrics import compute_metrics

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. ETL Execution

Run the ETL pipeline to produce train/val/test Parquet files.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '../data/etl/prepare_reviews.py', '--config', '../configs/data_config.yaml'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 2. Config Loading

In [ ]:
model_cfg = OmegaConf.load('../configs/model_config.yaml')
train_cfg = OmegaConf.load('../configs/training_config.yaml')
data_cfg  = OmegaConf.load('../configs/data_config.yaml')

print('Backbone:', model_cfg.model.backbone)
print('LoRA rank:', model_cfg.lora.r)
print('Epochs:', train_cfg.training.num_train_epochs)
print('Learning rate:', train_cfg.training.learning_rate)

## 3. Data Loading

In [ ]:
processed = Path('../data/processed')
train_df = pd.read_parquet(processed / 'train.parquet')
val_df   = pd.read_parquet(processed / 'val.parquet')

print(f'Train: {len(train_df):,} rows | positive: {train_df.label.mean():.1%}')
print(f'Val:   {len(val_df):,} rows | positive: {val_df.label.mean():.1%}')

tokenizer = AutoTokenizer.from_pretrained(model_cfg.model.backbone)
train_ds = DissatisfactionDataset(train_df, tokenizer, max_length=model_cfg.model.max_length)
val_ds   = DissatisfactionDataset(val_df, tokenizer, max_length=model_cfg.model.max_length)

## 4. Backbone + LoRA Instantiation

LoRA dramatically reduces the number of trainable parameters while preserving backbone expressiveness.

In [ ]:
base_model  = load_backbone(model_cfg)
peft_model  = apply_lora(base_model, model_cfg)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in peft_model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

# Show trainable layer names
print('\nTrainable layers:')
for name, p in peft_model.named_parameters():
    if p.requires_grad:
        print(' ', name)

## 5. Class Weight Computation

Natural class imbalance (~15–20% positive) requires class-weighted loss to prevent the model from predicting the majority class for all samples.

In [ ]:
class_weights = compute_class_weights(train_df['label'].tolist(), device=torch.device(DEVICE))
print(f'Class weights: negative={class_weights[0]:.4f}, positive={class_weights[1]:.4f}')

## 6. Training Run

In [ ]:
training_args = TrainingArguments(
    output_dir=train_cfg.training.output_dir,
    num_train_epochs=train_cfg.training.num_train_epochs,
    per_device_train_batch_size=train_cfg.training.per_device_train_batch_size,
    per_device_eval_batch_size=train_cfg.training.per_device_eval_batch_size,
    learning_rate=train_cfg.training.learning_rate,
    weight_decay=train_cfg.training.weight_decay,
    warmup_ratio=train_cfg.training.warmup_ratio,
    lr_scheduler_type=train_cfg.training.lr_scheduler_type,
    evaluation_strategy=train_cfg.training.evaluation_strategy,
    save_strategy=train_cfg.training.save_strategy,
    load_best_model_at_end=train_cfg.training.load_best_model_at_end,
    metric_for_best_model=train_cfg.training.metric_for_best_model,
    fp16=(DEVICE == 'cuda'),
    seed=train_cfg.training.seed,
    logging_steps=train_cfg.training.logging_steps,
    report_to=train_cfg.training.report_to,
)

trainer = WeightedLossTrainer(
    model=peft_model.to(DEVICE),
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[WandbMetricsCallback()],
    class_weights=class_weights,
)

trainer.train()

## 7. Checkpoint Selection

The best checkpoint is selected by **validation F1** (`metric_for_best_model: f1`). LoRA checkpoint is saved to `outputs/checkpoints/`. The adapter can be uploaded to HuggingFace Hub for zero-clone inference.

In [ ]:
# Save the best model's adapter
best_checkpoint = trainer.state.best_model_checkpoint
print(f'Best checkpoint: {best_checkpoint}')
print(f'Best val F1: {trainer.state.best_metric:.4f}')

# Save tokenizer alongside adapter for DissatisfactionPredictor
tokenizer.save_pretrained(best_checkpoint)

## Key Question

> *How much does LoRA reduce compute cost without sacrificing performance?*

LoRA reduces the number of trainable parameters to **< 1% of the backbone** (typically ~0.3% for DistilBERT with r=8). Full fine-tuning of DistilBERT requires updating ~66M parameters; LoRA updates only ~200k. Training time on an RTX 3060 drops from ~4 hours (full fine-tune) to ~35 minutes (LoRA), while validation F1 degrades by < 1 point in comparable setups. See `03_evaluation.ipynb` for a direct comparison in the baseline table.